# Notebook 2 — Thăm dò mô hình ngôn ngữ (zero-shot probing)

Notebook này đo xem một mô hình ngôn ngữ (LM) còn làm đúng **tác vụ** trên câu
phương ngữ hay không — không chỉ đo độ "quen" (perplexity). Cách tiếp cận lặp
lại logic của `src/probe_models.py` trong codebase nghiên cứu: với mỗi câu, ta
chấm điểm log-probability của **mỗi nhãn ứng viên** rồi lấy softmax trong tập
ứng viên. Kết quả là candidate-normalized score, không phải xác suất đã calibrated.

## Mục tiêu học tập

1. Phân biệt NLL familiarity probe với zero-shot task probing.
2. Viết được chấm điểm log-probability của một completion và softmax theo nhãn.
3. Đo **accuracy** và **gold candidate score** trên chuẩn vs. phương ngữ.
4. Diễn giải proxy score đúng giới hạn, không gọi là calibrated confidence.
5. Giải thích vì sao accuracy tuyệt đối thấp không loại trừ việc đo được degradation.

## Tham chiếu

Code gốc trong codebase: `src/probe_models.py` (chạy local LM) và
`src/probe_openai.py` (chạy GPT-4o qua API). Cả hai dùng chung logic: build
prompt → chấm điểm mỗi nhãn ứng viên → softmax trong candidate set → nhãn dự
đoán + proxy score. Notebook bắt buộc một LM nhỏ; hai model còn lại là bonus.

## Công thức

Với prompt `x` và một nhãn ứng viên `y` (được bọc thành JSON như `{"label":"Anger"}`):

```text
seq_logprob(y | x) = Σ log p(y_t | x, y_<t)
avg_logprob(y | x) = seq_logprob / |y|
CandidateScore(label | x) = softmax(avg_logprob) trong tập nhãn ứng viên
prediction = argmax_label CandidateScore(label | x)
proxy_confidence = CandidateScore(prediction | x)
gold_candidate_score = CandidateScore(gold | x)
```

**Degradation (accuracy)** = acc(standard) − acc(dialect).
**Gold-score erosion** =
mean_gold_candidate_score(standard) − mean_gold_candidate_score(dialect).

Lợi thế so với perplexity: metric này gắn với **tác vụ** (đúng/sai nhãn), nên
dễ diễn giải với người làm ứng dụng. Hạn chế: chỉ áp dụng cho task phân loại
có nhãn cố định (MCQA, NLI, SENT). QA là sinh tự do, để phần mở rộng.

In [1]:
from pathlib import Path
import sys

# Detect project root whether we run from notebooks/ or project root.
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "data" / "train_240.jsonl").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run the notebook from the project root or notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
print(f"Project root: {ROOT}")

Project root: /Users/totrinh/Dev/Dialectal-Robustness-of-LLMs-under-Meaning-Preserving-Vietnamese-Variation/seas_2026_student_project


In [2]:
import subprocess
import sys

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45,<5", "torch>=2.2,<3", "sentencepiece>=0.2,<1",
    ])

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from vialect_seas.data import DIALECTS, TASKS, load_jsonl
from vialect_seas.prompting import (
    LABEL_CANDIDATES, build_task_prompt, candidate_completion,
    is_classification, normalize_label, parse_prediction, gold_label,
)
from vialect_seas.probing import (
    DEFAULT_MODELS, MODEL_REVISIONS, load_text_generator, generate,
    score_completion, score_label_distribution,
    softmax_scores, probe_classification_rows,
)
from vialect_seas.metrics import paired_cluster_bootstrap

sns.set_theme(style="whitegrid", context="notebook")
train = load_jsonl(ROOT / "data" / "train_240.jsonl")
test = load_jsonl(ROOT / "data" / "test_300.jsonl")
print(f"train={len(train)}  test={len(test)}")
print("Classification tasks (có nhãn cố định):",
      [t for t in TASKS if is_classification(t)])
display(pd.DataFrame([
    {"model_id": model_id, "revision": MODEL_REVISIONS[model_id]}
    for model_id in DEFAULT_MODELS
]))

train=240  test=300
Classification tasks (có nhãn cố định): ['MCQA', 'NLI', 'SENT']


,model_id,revision
0,Qwen/Qwen2.5-0.5B,060db6499f32faf8b98477b0a26969ef7d8b9987
1,bigscience/bloom-560m,ac2ae5fab2ce3f9f40dc79b5ca9f637430d24971
2,VietAI/gpt-neo-1.3B-vietnamese-news,1be2f0c2e4193b525166f1286df874a0cadb0813


### STUDENT TASK 0 — Kế hoạch thí nghiệm (preregistration)

Trước khi chạy mô hình, ghi dự đoán có hướng. Giữ dự đoán này khi xem kết quả.

In [4]:
# HINT: RQ nên nêu rõ (1) input/population, (2) yếu tố so sánh, (3) metric.
"""Your code here"""
TEAM_NAME = "TODO"
PROBE_RQ = "TODO: ví dụ - ba LM có cùng suy giảm accuracy khi đổi chuẩn → dialect không?"
# Dự đoán thứ tự dialect từ ít → nhiều degradation.
EXPECTED_DIALECT_ORDER = ["TODO", "TODO", "TODO"]
# Dự đoán: gold-score erosion có cùng hướng với accuracy degradation không?
PRED_GOLD_SCORE_EROSION = "TODO: cùng hướng / ngược hướng / không liên quan"

print({
    "team": TEAM_NAME,
    "rq": PROBE_RQ,
    "expected_order": EXPECTED_DIALECT_ORDER,
    "pred_gold_score": PRED_GOLD_SCORE_EROSION,
})

{'team': 'TODO', 'rq': 'TODO: ví dụ - ba LM có cùng suy giảm accuracy khi đổi chuẩn → dialect không?', 'expected_order': ['TODO', 'TODO', 'TODO'], 'pred_gold_score': 'TODO: cùng hướng / ngược hướng / không liên quan'}


## Mô hình bắt buộc và phần bonus

| Mô hình | Đặc điểm |
|---|---|
| `Qwen/Qwen2.5-0.5B` | multilingual base LM, scorer nhỏ |
| `bigscience/bloom-560m` | multilingual causal LM, tokenizer khác Qwen |
| `VietAI/gpt-neo-1.3B-vietnamese-news` | LM chuyên biệt tiếng Việt / tin tức |

Học viên chỉ bắt buộc chạy `Qwen/Qwen2.5-0.5B`. BLOOM và Vietnamese GPT-Neo là
bonus để khảo sát tokenizer/model-family effects. Accuracy tuyệt đối có thể thấp;
primary comparison là paired degradation bên trong cùng một model.

## Chuẩn bị subset probing

Chỉ lấy task phân loại (MCQA, NLI, SENT) vì có nhãn ứng viên cố định. Lấy cân
bằng: `N_PER_CELL` câu/task/dialect từ train (giữ test nguyên để đánh giá sau).

In [5]:
N_PER_CELL = 3  # 3 câu/task/dialect → 3 tasks × 3 dialect × 3 = 27 cặp × 2 variant = 54 probe
RUN_PROBING = False  # Đổi True khi có GPU/runtime phù hợp.
RUN_BONUS_MODELS = False
REQUIRED_MODELS = [DEFAULT_MODELS[0]]
MODELS_TO_RUN = list(DEFAULT_MODELS) if RUN_BONUS_MODELS else REQUIRED_MODELS

classification_train = train[train["task"].map(is_classification)].copy()
probe_frame = (
    classification_train
    .sort_values(["task", "target_dialect", "sample_id"])
    .groupby(["task", "target_dialect"], observed=True, group_keys=False)
    .head(N_PER_CELL)
    .reset_index(drop=True)
)
print("Probe rows:", len(probe_frame))
print(probe_frame.groupby(["task", "target_dialect"], observed=True).size().unstack(fill_value=0))

Probe rows: 27
target_dialect  PNB  PNT2  PNT3
task                           
MCQA              3     3     3
NLI               3     3     3
SENT              3     3     3


### STUDENT TASK 1 — Kiểm tra prompt và nhãn ứng viên

Trước khi chạy mô hình, in prompt cho một câu SENT và một câu NLI để xác nhận
prompt đúng định dạng. Kiểm tra nhãn gold đã được chuẩn hóa đúng.

In [6]:
RUN_PROMPT_CHECK = False

if RUN_PROMPT_CHECK:
    # HINT 1: chọn 1 row SENT và 1 row NLI từ probe_frame.
    # HINT 2: in build_task_prompt(row, variant="dialect") và gold_label(row).
    # HINT 3: kiểm tra gold_label nằm trong LABEL_CANDIDATES[probe_task(row["task"])].
    """Your code here"""
    sent_row = None
    nli_row = None

    # SELF-CHECK
    assert sent_row is not None and nli_row is not None
    from vialect_seas.prompting import probe_task
    for r in [sent_row, nli_row]:
        g = gold_label(r)
        cands = LABEL_CANDIDATES[probe_task(r["task"])]
        assert g in cands, f"gold {g!r} không nằm trong {cands}"
    print("Prompt checks passed.")

## Chạy zero-shot probing

Code tải từng LM, chấm điểm mỗi nhãn ứng viên cho mỗi (row × variant), rồi giải
phóng VRAM trước khi tải LM tiếp theo. Bắt đầu với subset nhỏ; tăng `N_PER_CELL`
sau khi đã kiểm tra thời gian và output.

In [7]:
from vialect_seas.prompting import probe_task
output_path = ROOT / "outputs" / "zero_shot_probe.csv"
output_path.parent.mkdir(exist_ok=True)

if RUN_PROBING:
    all_results = []
    for model_id in MODELS_TO_RUN:
        print(f"=== Probing {model_id} ===", flush=True)
        runner = load_text_generator(model_id)
        result = probe_classification_rows(probe_frame, runner, variants=("standard", "dialect"))
        all_results.append(result)
        # Giải phóng VRAM.
        import gc, torch
        del runner
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    probe_scores = pd.concat(all_results, ignore_index=True)
    probe_scores.to_csv(output_path, index=False)
    print("Saved", output_path)
elif output_path.exists():
    probe_scores = pd.read_csv(output_path)
    print("Loaded existing", output_path)
else:
    probe_scores = pd.DataFrame()
    print("Set RUN_PROBING=True (cần GPU) để chạy probing")

Set RUN_PROBING=True (cần GPU) để chạy probing


### Sanity checks

1. Mỗi mô hình có cùng số dòng (row × variant).
2. Không có proxy score NaN hoặc ngoài [0, 1].
3. Số nhãn dự đoán.unique hợp lý (giới hạn trong candidates).

In [8]:
if not probe_scores.empty:
    checks = probe_scores.groupby("model_id").agg(
        rows=("sample_id", "size"),
        mean_proxy=("proxy_confidence", "mean"),
        min_proxy=("proxy_confidence", "min"),
        max_proxy=("proxy_confidence", "max"),
    )
    display(checks.round(4))
    assert checks["rows"].nunique() == 1, "Số dòng không khớp giữa các mô hình"
    assert checks["min_proxy"].ge(0).all() and checks["max_proxy"].le(1.001).all()

## Accuracy degradation theo dialect

`Drop = acc(standard) − acc(dialect)`; số dương nghĩa là mô hình **giảm đúng**
trên dialect. So sánh drop giữa ba mô hình và ba dialect.

In [9]:
if not probe_scores.empty:
    acc = (
        probe_scores.groupby(["model_id", "target_dialect", "variant"], observed=True)
        .agg(accuracy=("correct", "mean"), n=("sample_id", "size"))
        .reset_index()
    )
    acc_pivot = acc.pivot_table(
        index=["model_id", "target_dialect"], columns="variant", values="accuracy"
    )
    acc_pivot["drop"] = acc_pivot["standard"] - acc_pivot["dialect"]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    plot_acc = acc_pivot.reset_index().melt(
        id_vars=["model_id", "target_dialect"],
        value_vars=["standard", "dialect"], var_name="variant", value_name="accuracy"
    )
    sns.barplot(data=plot_acc, x="target_dialect", y="accuracy", hue="variant", ax=ax)
    ax.set(title="Accuracy: chuẩn vs. dialect (zero-shot)",
           xlabel="Dialect", ylabel="Accuracy")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
    display(acc_pivot.round(3))

    accuracy_ci = paired_cluster_bootstrap(
        probe_scores,
        value_column="correct",
        group_by=["model_id", "target_dialect"],
        n_resamples=2000,
        seed=2026,
    )
    accuracy_ci["metric"] = "accuracy_drop"
    display(accuracy_ci.round(4))

### Insight (accuracy)

Viết 3–5 câu theo cấu trúc observation → evidence → caveat:

- Accuracy tuyệt đối có thể thấp (zero-shot, LM nhỏ) — điều đó không loại trừ
  việc đo được degradation có hướng.
- Dialect nào có drop lớn nhất? Có nhất quán giữa ba mô hình không?
- **Caveat:** subset nhỏ (N_PER_CELL=3) → khoảng tin cậy rộng; cần bootstrap
  theo source để báo cáo uncertainty.

## Gold candidate score erosion

Primary analysis dùng score của **nhãn gold**, không dùng score của nhãn model tự
dự đoán. Mô hình có thể rất chắc vào một nhãn sai; proxy confidence cao khi đó
không phải bằng chứng robustness. Các score chỉ được chuẩn hóa trong candidate set.

In [10]:
if not probe_scores.empty:
    score_summary = (
        probe_scores.groupby(["model_id", "target_dialect", "variant"], observed=True)
        .agg(mean_gold_score=("gold_candidate_score", "mean"),
             mean_proxy_confidence=("proxy_confidence", "mean"))
        .reset_index()
    )
    gold_pivot = score_summary.pivot_table(
        index=["model_id", "target_dialect"], columns="variant", values="mean_gold_score"
    )
    gold_pivot["gold_score_erosion"] = gold_pivot["standard"] - gold_pivot["dialect"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    plot_score = gold_pivot.reset_index().melt(
        id_vars=["model_id", "target_dialect"],
        value_vars=["standard", "dialect"], var_name="variant", value_name="mean_gold_score"
    )
    sns.barplot(data=plot_score, x="target_dialect", y="mean_gold_score",
                hue="variant", ax=axes[0])
    axes[0].set(title="Gold candidate score: chuẩn vs. dialect", xlabel="Dialect",
                ylabel="Mean gold candidate score")
    axes[0].set_ylim(0, 1)

    sns.heatmap(gold_pivot["gold_score_erosion"].unstack(), annot=True, fmt=".3f",
                cmap="RdBu_r", center=0, ax=axes[1])
    axes[1].set(title="Gold-score erosion (chuẩn − dialect)",
                xlabel="Dialect", ylabel="Model")
    plt.tight_layout()
    plt.show()
    display(gold_pivot.round(3))

    gold_score_ci = paired_cluster_bootstrap(
        probe_scores,
        value_column="gold_candidate_score",
        group_by=["model_id", "target_dialect"],
        n_resamples=2000,
        seed=2026,
    )
    gold_score_ci["metric"] = "gold_score_erosion"
    display(gold_score_ci.round(4))

### Insight (candidate score)

- Gold-score erosion có cùng hướng với accuracy degradation không?
- Có trường hợp accuracy không đổi nhưng gold score giảm không?
- **Caveat:** đây là candidate-normalized proxy, không phải calibrated probability.
  Score còn có thể nhạy với cách viết và độ dài completion JSON.

In [11]:
# So sánh theo task: degradation có khác giữa MCQA / NLI / SENT không?
if not probe_scores.empty:
    task_acc = (
        probe_scores.groupby(["model_id", "task", "variant"], observed=True)
        .agg(accuracy=("correct", "mean"))
        .reset_index()
    )
    task_acc_pivot = task_acc.pivot_table(
        index=["model_id", "task"], columns="variant", values="accuracy"
    )
    task_acc_pivot["drop"] = task_acc_pivot["standard"] - task_acc_pivot["dialect"]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    sns.heatmap(task_acc_pivot["drop"].unstack(), annot=True, fmt=".3f",
                cmap="RdBu_r", center=0, ax=ax)
    ax.set(title="Accuracy drop theo task (chuẩn − dialect)",
           xlabel="Task", ylabel="Model")
    plt.tight_layout()
    plt.show()
    display(task_acc_pivot.round(3))

### Insight (theo task)

Xác định task nào tạo degradation lớn nhất. Với MCQA, kiểm tra liệu rewrite
có giữ đủ 4 lựa chọn và đáp án đúng không (xem Notebook 1). Với NLI, premise
không đổi — chỉ hypothesis bị dialect hóa — nên degradation cô lập được yếu tố
hypothesis.

## Bài tập mở

1. Tăng `N_PER_CELL` và so sánh độ rộng bootstrap CI theo `sample_id`.
2. So sánh thứ tự khó ở đây với degradation trong Notebook 1 (10 mô hình).
   **Correlation ≠ causation** — giải thích vì sao.
3. QA là task sinh tự do. Dùng `generate()` để sinh câu trả lời, rồi tính
   exact-match / contains-match với gold. So sánh chuẩn vs. dialect.
4. Chạy hai bonus model và so sánh tokenizer/model-family effects.
5. Kiểm tra 10 câu có gold-score erosion lớn nhất; phân loại nguyên nhân
   (từ vựng, ngữ pháp, độ dài).

In [12]:
# STUDENT TASK 2 (EXTENSION) — Một thí nghiệm probing mở.
RUN_STUDENT_PROBE = False

def student_probe_analysis(scores):
    # HINT 1: chọn một biến chưa thử (task subset, normalized_direct, QA generative).
    # HINT 2: resample theo sample_id, giữ ba dialect của source trong cluster.
    # HINT 3: báo mean, 95% interval, n và aggregation unit.
    """Your code here"""
    return None

if RUN_STUDENT_PROBE:
    result = student_probe_analysis(probe_scores)
    if result is None:
        raise NotImplementedError("Hoàn thành student_probe_analysis trước khi bật cờ")
    # SELF-CHECK: bảng phải có model × dialect và interval hợp lệ.
    required = {"model_id", "target_dialect", "mean", "ci_low", "ci_high", "n_sources"}
    assert required.issubset(result.columns)
    assert (result["ci_low"] <= result["mean"]).all()
    assert (result["mean"] <= result["ci_high"]).all()
    display(result)